# Image Captioning Using Tika’s Show and Tell Caption Generator

This script generates AI-generated captions for images of 10,000 haunted places. Each image is sent to a locally running Tika im2txt-rest-tika Docker container, which hosts the Show and Tell neural image captioning model.

The service is accessed through the /inception/v3/caption/image endpoint, which internally uses the InceptionV3 model for feature extraction and the Show and Tell model for sentence generation.

This script was run in batches, adjusting the range of IDs each time (e.g., 0–2000, 2000–4000, ...), but ultimately generated captions for the first 10,000 haunted places (IDs 0–9999).

For each image:
- The script checks if a caption already exists in the dataset.
- If not, it posts the image to the captioning API.
- It extracts the caption with the highest confidence score.
- The caption is stored in the `Image_Caption` column of the dataset.

The final dataset (`haunted_places_features_added_v2.tab`) is updated and saved with all newly generated captions.

In [24]:
import os
import requests
import pandas as pd

# File paths
DATASET_PATH = "../data/processed/haunted_places_features_added_v2.tab"
IMAGE_DIR = "../data/generated_images/"
IM2TXT_URL = "http://localhost:8764/inception/v3/caption/image"

# Load dataset
df = pd.read_csv(DATASET_PATH, sep="\t")

for idx in range(8000, 10000): # I adjusted range based caption batch I wanted to produce
    if idx >= len(df):
        break

    if pd.notna(df.at[idx, "Image_Caption"]) and df.at[idx, "Image_Caption"] != "":
        print(f"Row {idx} already has caption, skipping.")
        continue

    img_path = os.path.join(IMAGE_DIR, f"hpimg_{idx}.png")
    if not os.path.exists(img_path):
        print(f"Missing image: {img_path}")
        continue

    try:
        with open(img_path, "rb") as f:
            response = requests.post(
                IM2TXT_URL,
                headers={"Content-Type": "image/jpeg"},
                data=f,
                timeout=10
            )
        if response.status_code == 200:
            result = response.json()
            captions = result.get("captions", [])
            if captions:
                # Get the caption with the highest confidence score
                best_caption = max(captions, key=lambda x: x["confidence"])
                caption = best_caption["sentence"]
                
                # Save only the best caption
                df.at[idx, "Image_Caption"] = caption
                print(f"Row {idx}: {caption}")
            else:
                print(f"No captions found for {img_path}")
        else:
            print(f"Error {response.status_code} on {img_path}")
    except Exception as e:
        print(f"Exception on {img_path}: {e}")

# Save updated file
df.to_csv(DATASET_PATH, sep="\t", index=False)
print("Dataset updated with best captions.")

Row 8000: a black and white photo of a man and a woman
Row 8001: a group of people sitting around a table in a room .
Row 8002: a room filled with lots of different colored lights .
Row 8003: a woman standing in front of a mirror .
Row 8004: a man is standing in front of a christmas tree .
Row 8005: a black and white photo of a street sign
Row 8006: a group of people sitting on top of a wooden bench .
Row 8007: a room filled with lots of wooden furniture .
Row 8008: a train traveling down tracks next to a tall building .
Row 8009: a very big pretty clock in a big room .
Row 8010: a black and white photo of people on a street .
Row 8011: a group of people standing on top of a building .
Row 8012: a display case in a store filled with lots of bottles .
Row 8013: a train station with a train on the tracks .
Missing image: ../data/generated_images/hpimg_8014.png
Row 8015: a black and white photo of a group of people on a street .
Row 8016: a clock tower in the middle of a city .
Row 8017: 